![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/01.9.PretrainedZeroShotMultiTask.ipynb)

If you are using the `spark-nlp-jsl` library, please use this [1.8.PretrainedZeroShotMultiTask](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/1.8.PretrainedZeroShotMultiTask.ipynb) notebook.

# 📌 **Pretrained Zero-Shot Multi-Task Learning in Spark NLP**

## 📌 **Healthcare NLP for Data Scientists Course**

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each components.

📚 `PretrainedZeroShotMultiTask` is a unified annotator in Spark NLP for Healthcare that performs **multiple NLP tasks simultaneously** in a single pipeline pass, without requiring any task-specific fine-tuning or labeled training data. It leverages advanced pretrained language models to handle Named Entity Recognition (NER), Text Classification, Relation Extraction, and Structured Extraction all at once.

📚 Traditional NLP pipelines require separate models for each task: one for NER, another for classification, another for relation extraction, and so on. `PretrainedZeroShotMultiTask` eliminates this complexity by combining all these capabilities into a single annotator. You simply define your entity labels, classification categories, relation types, and structured fields through setter methods, and the model handles everything in one forward pass.

📚 This approach is especially useful in clinical and healthcare NLP workflows where you need to extract entities (e.g., problems, treatments, drugs), classify documents (e.g., by urgency or cancer type), detect relationships between entities (e.g., a treatment administered for a problem), and capture structured information (e.g., assertion status of a condition) all from the same text.

📚 In this notebook, you will learn how to use `PretrainedZeroShotMultiTask` to configure and run multi-task pipelines, interpret the different output annotation types, use entity descriptions for improved extraction quality, and integrate with downstream annotators like `MultiAnnotationSplitter` and `LightDeIdentification`.

- 🪅 **Pretrained Zero-Shot Multi-Task Models**

| # | Model | Description |
|--:|:------|:------------|
| 1 | [zeroshot_multitask_base](https://nlp.johnsnowlabs.com/2026/04/10/zeroshot_multitask_base_en.html) | Base model for zero-shot multi-task extraction across NER, classification, relation, and structured tasks |
| 2 | [zeroshot_multitask_generic](https://nlp.johnsnowlabs.com/2026/04/09/zeroshot_multitask_generic_en.html) | Generic model with broader coverage, recommended for complex clinical scenarios |

- 🪅 **Key Parameters**

| Parameter | Description |
|:----------|:------------|
| `setEntities(list)` | Define NER entity labels to extract. Supports `LABEL::description` format for guided extraction (e.g., `"DRUG::medication name like aspirin"`). |
| `setClassifications(list)` | Define classification tasks as `(task_name, [labels])` tuples. Each task independently assigns one label per document. |
| `setRelations(list)` | Define relation types to detect between extracted entities (e.g., `"treatment_administered_for_problem"`). |
| `setStructures(list)` | Define structured extraction templates as `(name, [field_definitions])` tuples. Field definitions use the `field_name::type::description` format. |
| `setEntityThreshold(float)` | Minimum confidence score to include an entity extraction. Higher values return fewer but more confident results. Default: `0.4`. |

- 🪅 **Output Annotation Types**

All task outputs are returned in a single column. Each annotation has an `annotatorType` field that indicates the task it belongs to:

| annotatorType | Task | Description |
|:--------------|:-----|:------------|
| `chunk` | NER | Extracted entity spans with `entity`, `confidence`, and character offsets |
| `category` | Classification / Relation | Classification results include a `task` field in metadata; relation results do not |
| `struct` | Structured Extraction | Structured fields returned as key-value pairs in metadata |

📌 **Related Notebooks:**
- [01.4.ZeroShot_Clinical_NER](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/01.4.ZeroShot_Clinical_NER.ipynb): Zero-Shot NER with `PretrainedZeroShotNER` (single-task NER only)
- [04.0.Clinical_DeIdentification](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/04.0.Clinical_DeIdentification.ipynb): De-identification pipelines using `LightDeIdentification`
- [02.0.Clinical_Assertion_Model](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/02.0.Clinical_Assertion_Model.ipynb): Assertion status detection for clinical entities

## Colab Setup

In [ ]:
# Install the johnsnowlabs library to access Spark-NLP for Healthcare.
! pip install -q johnsnowlabs

In [2]:
from google.colab import files

print('Please Upload your John Snow Labs License using the button below')
license_keys = files.upload()

Please Upload your John Snow Labs License using the button below


Saving spark_jsl.json to spark_jsl.json


In [ ]:
from johnsnowlabs import nlp, medical

# After uploading your license run this to install all licensed Python Wheels and pre-download Jars the Spark Session JVM
nlp.settings.enforce_versions=False
nlp.install(refresh_install=True)

In [4]:
from johnsnowlabs import nlp, medical
from pyspark.sql import functions as F

import warnings
warnings.filterwarnings('ignore')

# Automatically load license data and start a session with all jars user has access to
spark = nlp.start()
spark.sparkContext.setLogLevel("ERROR")
spark

👌 Detected license file /content/spark_jsl.json
👌 Launched cpu optimized session with with: 🚀Spark-NLP==6.4.0, 💊Spark-Healthcare==6.4.0, running on ⚡ PySpark==3.4.0


# 1. `zeroshot_multitask_base`

`zeroshot_multitask_base` is the base pretrained model for zero-shot multi-task extraction. It supports all four task types (NER, Classification, Relation Extraction, and Structured Extraction) and provides a good balance between speed and accuracy.

**Let's load the model.**

In [5]:
model = medical.PretrainedZeroShotMultiTask.pretrained("zeroshot_multitask_base", "en", "clinical/models")

zeroshot_multitask_base download started this may take some time.
Approximate size to download 805.4 MB
[OK!]


## 1.1. Basic Multi-Task Pipeline (NER + Classification + Relation + Struct)

In this section, we create a pipeline that performs all four tasks at once on a sample text. We configure:

- **NER** with `setEntities()` to extract `person`, `organization`, and `location` entities
- **Classification** with `setClassifications()` to classify sentiment as positive, negative, or neutral
- **Relation Extraction** with `setRelations()` to detect `works_for` relationships between entities
- **Structured Extraction** with `setStructures()` to extract price information with a monetary value field

`PretrainedZeroShotMultiTask` supports the `LABEL::description` format when defining entities via `setEntities()`. By providing a natural language description alongside the label, you guide the model to better understand what each entity type represents.

**Syntax:** `"LABEL::description of what to extract"`

For example:
- `"NAME::name of person"` tells the model to look for person names
- `"DOCTOR::name of doctor"` tells the model to specifically look for doctor names
- `"DRUG::medication name like aspirin"` gives the model an example to calibrate its understanding

This is especially helpful when you need fine-grained entity types that go beyond generic labels.

**Let's create a sample dataframe and build the pipeline.**

In [6]:
cancer_text = """The had previously undergone a left mastectomy and an axillary lymph node dissection for a left breast cancer twenty years ago.
The tumor was positive for ER and PR. Postoperatively, radiotherapy was administered to her breast.The cancer recurred as a right lung metastasis 13 years later.
The patient underwent a regimen consisting of adriamycin (60 mg/m2) and cyclophosphamide (600 mg/m2) over six courses, as first line therapy."""

data = spark.createDataFrame([[cancer_text]]).toDF("text")

document_assembler = (
   nlp.DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

zero_shot = (
    model
    .setInputCols(["document"])
    .setOutputCol("extractions")
    .setEntities([
            "Cancer_Dx::The exact cancer diagnosis phrase, such as 'left breast cancer' or 'lung cancer'. Extract only the diagnosis phrase itself.",
            "Site::The exact anatomical site phrase, including laterality when present, such as 'left breast' or 'right lung'. Extract only the site phrase itself.",
            "Biomarker::The exact biomarker name, such as 'ER', 'PR', 'HER2', or 'PD-L1'. Extract only the marker name itself.",
            "Biomarker_Result::The exact biomarker result term, such as 'positive', 'negative', 'amplified', 'high', or 'low'. Extract only the result term itself.",
            "Metastasis::The exact metastatic disease phrase including site when present, such as 'right lung metastasis' or 'bone metastasis'. Extract only the metastasis phrase itself.",
            "Cancer_Surgery::The exact cancer surgery phrase, including laterality when present, such as 'left mastectomy' or 'axillary lymph node dissection'. Extract only the procedure phrase itself.",
            "Chemotherapy::The exact chemotherapy drug name or treatment term, such as 'adriamycin', 'cyclophosphamide', or 'chemotherapy'. Extract only the drug or treatment term itself.",
            "Radiotherapy::The exact radiation treatment term, such as 'radiotherapy' or 'radiation therapy'. Extract only the treatment term itself, not surrounding action phrases, recipients, or body sites.",
            "Line_Of_Therapy::The exact treatment-line phrase, such as 'first line therapy' or 'second line treatment'. Extract only the treatment-line phrase itself.",
            "Response_To_Treatment::The exact response or progression term, such as 'recurred', 'progressed', 'improved', or 'stable disease'. Extract only the response term itself.",
            "Dosage::The exact dose expression, such as '60 mg/m2' or '400 mg'. Extract only the dose expression itself.",
            "Relative_Date::The exact relative time phrase, such as 'twenty years ago' or '13 years later'. Extract only the temporal phrase itself."
    ]
         )
    .setStructures([
    ("cancer_case", [
        "diagnosis::str::The exact cancer diagnosis phrase, such as 'left breast cancer' or 'lung cancer'. Extract only the diagnosis phrase itself.",
        "site::list::The exact anatomical site phrase related to the cancer, including laterality when present, such as 'left breast' or 'right lung'. Extract only the site phrase itself, not the diagnosis phrase.",
        "metastasis::list::The exact metastatic disease phrase, including site when present, such as 'right lung metastasis' or 'bone metastasis'. Extract only the metastasis phrase itself.",
        "surgery::list::The exact cancer-related surgery phrase, including laterality when present, such as 'left mastectomy' or 'axillary lymph node dissection'. Extract only the procedure phrase itself, not surrounding action phrases.",
        "radiotherapy::list::The exact radiation treatment term, such as 'radiotherapy' or 'radiation therapy'. Extract only the treatment term itself, not surrounding action phrases, recipients, or body sites.",
        "line_of_therapy::str::The exact treatment-line phrase, such as 'first line therapy' or 'second line therapy'. Extract only the treatment-line phrase itself.",
        "response::str::The exact response or progression term, such as 'recurred', 'progressed', 'improved', or 'stable disease'. Extract only the response term itself.",
        "relative_date::list::The exact relative time phrase, such as 'twenty years ago' or '13 years later'. Extract only the temporal phrase itself."
    ]),
    ("biomarker_item", [
        "name::str::The exact biomarker name, such as 'ER', 'PR', 'HER2', or 'PD-L1'. Extract only the biomarker name itself.",
        "result::str::The exact biomarker result term associated with that biomarker, such as 'positive', 'negative', 'amplified', 'high', or 'low'. Extract only the result term itself."
    ]),
    ("chemotherapy_item", [
        "drug::str::The exact chemotherapy drug name, such as 'adriamycin' or 'cyclophosphamide'. Extract only the drug name itself.",
        "dosage::str::The exact dose expression associated with the drug, such as '60 mg/m2' or '600 mg/m2'. Extract only the dosage expression itself."
    ])
])
    .setEntityThreshold(0.4)
)

pipe_model = nlp.Pipeline().setStages([document_assembler, zero_shot]).fit(data)

results = (
    pipe_model
    .transform(data)
).cache()

collected = results.select("extractions").collect()

results.selectExpr("explode(extractions) as extractions").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Let's parse and display the results for each task type.**

## Helper Functions for Parsing Multi-Task Results

Since `PretrainedZeroShotMultiTask` returns all task outputs (NER chunks, classifications, relations, structures) in a single `extractions` column, we define a helper function that splits them by `annotatorType` and returns organized DataFrames for each task.

In [7]:
def parse_multitask_results(results_df, extractions_col="extractions"):
    """Parse multi-task extraction results into separate DataFrames by annotation type."""

    collected = results_df.select(extractions_col).collect()

    chunk_list = []
    category_list = []
    relation_list = []
    struct_list = []

    for idx, row in enumerate(collected):
        for ext in getattr(row, extractions_col):

            if ext.annotatorType == "chunk":
                row_dict = {"idx": idx, "begin": ext.begin, "end": ext.end, "chunk": ext.result}
                row_dict.update(ext.metadata)
                chunk_list.append(row_dict)

            elif ext.annotatorType == "category":
                row_dict = {"idx": idx, "begin": ext.begin, "end": ext.end, "result": ext.result}
                row_dict.update(ext.metadata)
                if ext.metadata.get("task"):
                    category_list.append(row_dict)
                else:
                    relation_list.append(row_dict)

            elif ext.annotatorType == "struct":
                row_dict = {"idx": idx, "begin": ext.begin, "end": ext.end, "result": ext.result}
                try:
                    row_dict.update(eval(ext.metadata.get("text", "{}")))
                except:
                    row_dict["raw_text"] = ext.metadata.get("text", "")
                struct_list.append(row_dict)

    dfs = {
        "ner_chunks": pd.DataFrame(chunk_list),
        "classifications": pd.DataFrame(category_list),
        "relations": pd.DataFrame(relation_list),
        "structures": pd.DataFrame(struct_list),
    }

    # Sort NER chunks if present
    if not dfs["ner_chunks"].empty:
        dfs["ner_chunks"] = dfs["ner_chunks"].sort_values(by=["idx", "begin", "end"]).reset_index(drop=True)

    return dfs

In [8]:
import json

def safe_get_json(meta, key):
    import json
    if key not in meta or meta[key] is None:
        return None
    return json.loads(meta[key])

def extract_value(meta_json):
    if isinstance(meta_json, dict):
        return meta_json.get("text")
    return None


output = {}

for r in collected:
    for e in r.extractions:

        # STRUCT TYPE
        if e.annotatorType == "struct":

            if e.result == "cancer_case":
                mc = output.setdefault("cancer_case", [{}])[0]
                meta = e.metadata

                mc["diagnosis"] = extract_value(safe_get_json(meta, "diagnosis"))
                mc["site"] = [x["text"] for x in safe_get_json(meta, "site") or []]
                mc["metastasis"] = [x["text"] for x in safe_get_json(meta, "metastasis") or []]
                mc["surgery"] = [x["text"] for x in safe_get_json(meta, "surgery") or []]
                mc["radiotherapy"] = [x["text"] for x in safe_get_json(meta, "radiotherapy") or []]
                mc["line_of_therapy"] = extract_value(safe_get_json(meta, "line_of_therapy"))
                mc["response"] = extract_value(safe_get_json(meta, "response"))
                mc["relative_date"] = [x["text"] for x in safe_get_json(meta, "relative_date") or []]

            elif e.result == "biomarker_item":
                output.setdefault("biomarker_item", []).append({
                    "name": extract_value(json.loads(e.metadata["name"])),
                    "result": extract_value(json.loads(e.metadata["result"]))
                })

            elif e.result == "chemotherapy_item":
                output.setdefault("chemotherapy_item", []).append({
                    "drug": extract_value(json.loads(e.metadata["drug"])),
                    "dosage": extract_value(json.loads(e.metadata["dosage"]))
                })

print(json.dumps(output, indent=2))

{
  "cancer_case": [
    {
      "diagnosis": "left breast cancer",
      "site": [
        "left breast"
      ],
      "metastasis": [
        "right lung metastasis"
      ],
      "surgery": [
        "left mastectomy",
        "axillary lymph node dissection"
      ],
      "radiotherapy": [
        "radiotherapy"
      ],
      "line_of_therapy": "first line therapy",
      "response": "recurred",
      "relative_date": [
        "13 years later",
        "twenty years ago"
      ]
    }
  ],
  "biomarker_item": [
    {
      "name": "ER",
      "result": "positive"
    },
    {
      "name": "PR",
      "result": "positive"
    }
  ],
  "chemotherapy_item": [
    {
      "drug": "adriamycin",
      "dosage": "60 mg/m2"
    },
    {
      "drug": "cyclophosphamide",
      "dosage": "600 mg/m2"
    }
  ]
}


In [10]:
import pandas as pd


dfs = parse_multitask_results(results)

print("=" * 60)
print("NER CHUNKS")
print("=" * 60)
display(dfs["ner_chunks"])

print("\n" + "=" * 60)
print("CLASSIFICATIONS")
print("=" * 60)
display(dfs["classifications"])

print("\n" + "=" * 60)
print("RELATIONS")
print("=" * 60)
display(dfs["relations"])

print("\n" + "=" * 60)
print("STRUCTURES")
print("=" * 60)
display(dfs["structures"])

NER CHUNKS


,idx,begin,end,chunk,sentence,ner_source,entity,confidence
0,0,54,83,axillary lymph node dissection,0,extractions,Cancer_Surgery,0.99978805
1,0,91,108,left breast cancer,0,extractions,Cancer_Dx,0.9994559
2,0,110,125,twenty years ago,0,extractions,Relative_Date,0.99857783
3,0,142,149,positive,0,extractions,Biomarker_Result,0.999972
4,0,162,163,PR,0,extractions,Biomarker,0.9999351
5,0,183,194,radiotherapy,0,extractions,Radiotherapy,0.9999644
6,0,238,245,recurred,0,extractions,Response_To_Treatment,0.99999183
7,0,252,272,right lung metastasis,0,extractions,Metastasis,0.9997722
8,0,336,345,adriamycin,0,extractions,Chemotherapy,0.99899733
9,0,380,388,600 mg/m2,0,extractions,Dosage,0.9999985



CLASSIFICATIONS


""



RELATIONS


""



STRUCTURES


,idx,begin,end,result
0,0,0,430,cancer_case
1,0,0,430,biomarker_item
2,0,0,430,biomarker_item
3,0,0,430,chemotherapy_item
4,0,0,430,chemotherapy_item


## 1.2. Using Entity Descriptions for Better Extraction



**Let's update the entity definitions with descriptions and re-run the pipeline.**

# 2. Downstream Integration

## 2.1. `MultiAnnotationSplitter` for Splitting Extractions

Since `PretrainedZeroShotMultiTask` returns all task outputs in a single `extractions` column with mixed annotation types (`chunk`, `category`, `struct`), you need `MultiAnnotationSplitter` to separate them before feeding into downstream annotators.

`MultiAnnotationSplitter` filters annotations by their `annotatorType` and outputs only the selected type. Setting `.setSplitType("chunk")` extracts only the NER chunk annotations, which can then be used by annotators that expect NER chunks as input (e.g., `LightDeIdentification`, `AssertionDLModel`, `RelationExtractionModel`).

**Let's build a pipeline with `MultiAnnotationSplitter` to extract NER chunks.**

In [11]:
# Rebuild the full multi-task pipeline
zero_shot_full = (
    model
    .setInputCols(["document"])
    .setOutputCol("extractions")
    .setEntities(["NAME::name of person", "LOCATION::city or place name", "ORGANIZATION::company name"])
)

multi_annotation_splitter = (
    medical.MultiAnnotationSplitter()
    .setInputCols("extractions")
    .setOutputCol("ner_chunk")
    .setSplitType("chunk")
)

full_pipe = nlp.Pipeline().setStages([
    document_assembler,
    zero_shot_full,
    multi_annotation_splitter,
]).fit(data)

split_results = full_pipe.transform(data)
split_results.select("ner_chunk").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ner_chunk                                                                                                                                                                                                                                            |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[{chunk, 252, 261, right lung, {sentence -> 0, entity -> LOCATION, confidence -> 0.5089722, ner_source -> extractions}, []}, {chunk, 336, 345, adriamycin, {sentence -> 0, entity -> NAME, confidence -> 0.88914394, ner_source -> extractions}, []}]|
+-------

## 2.2. De-identification with `LightDeIdentification`

Once NER chunks are extracted via `MultiAnnotationSplitter`, they can be directly fed into `LightDeIdentification` to mask or obfuscate protected health information (PHI). This enables a complete zero-shot de-identification workflow without any pretrained NER model or embeddings.

Healthcare NLP **6.4.0** adds **`setDeidMarkers`**, which wraps de-identified spans with configurable prefix/suffix markers in the output text (for example `{"start": "[S]", "end": "[E]"}`).

For more details on de-identification, see [04.0.Clinical_DeIdentification](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/04.0.Clinical_DeIdentification.ipynb).

**Let's add `LightDeIdentification` to the pipeline.**

In [12]:
light_deIdentification = (
    medical.LightDeIdentification()
    .setInputCols("document", "ner_chunk")
    .setOutputCol("deid")
    .setDeidMarkers({"start": "[S]", "end": "[E]"})
)

deid_pipe = nlp.Pipeline().setStages([
    document_assembler,
    zero_shot_full,
    multi_annotation_splitter,
    light_deIdentification,
]).fit(data)

deid_results = deid_pipe.transform(data)
deid_results.select("deid.result").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|result                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
+-----------------------------------------------------------------------------------------------------------

## 2.3. LightPipeline for Quick Inference

`LightPipeline` wraps a fitted `PipelineModel` and provides fast, single-record inference without Spark overhead. This is useful for testing, prototyping, and serving predictions in real-time applications.

**Let's create a `LightPipeline` and run a quick test.**

In [13]:
light_pipe = nlp.LightPipeline(deid_pipe)
light_result = light_pipe.annotate(
    "Dr. Sarah Johnson works at Mayo Clinic in Rochester. "
    "The consultation fee is $250. Sentiment: very satisfied."
)

for key in ["ner_chunk", "deid"]:
    print(f"{key}: {light_result.get(key, [])}")

ner_chunk: ['Dr. Sarah Johnson', 'Mayo Clinic', 'Rochester']
deid: ['[S]<NAME>[E] works at [S]<ORGANIZATION>[E] in [S]<LOCATION>[E]. The consultation fee is $250. Sentiment: very satisfied.']


# 3. `zeroshot_multitask_generic`

`zeroshot_multitask_generic` is a broader, more capable model that provides better coverage for complex clinical scenarios. It is recommended when you need to run multiple task types simultaneously on diverse clinical texts, especially when combining NER with assertion detection (via structured extraction), document classification, and relation extraction.

**Let's load the generic model.**

In [14]:
generic = medical.PretrainedZeroShotMultiTask.pretrained("zeroshot_multitask_generic", "en", "clinical/models")

zeroshot_multitask_generic download started this may take some time.
Approximate size to download 805.4 MB
[OK!]


## 3.1. Comprehensive Clinical Multi-Task Example

In this section, we demonstrate a comprehensive clinical pipeline that performs:

- **NER** to extract problems, treatments, tests, body parts, drug dosages, severity indicators, and dates
- **Structured Extraction** to capture assertion status (`present`, `absent`, `hypothetical`, `possible`, `conditional`, `associated_with_someone_else`) for problems and treatments
- **Document Classification** to detect cancer type and urgency level
- **Relation Extraction** to identify clinical relationships like `treatment_administered_for_problem`, `treatment_causes_problem`, and `test_reveals_problem`

We use four sample clinical texts covering different scenarios: patient history, medication list, lab findings, and oncology report.

**Let's create a dataframe with sample clinical texts.**

In [15]:
sample_texts = [
    (
        "Jennifer Smith is a 28-year-old female with a history of gestational diabetes mellitus "
        "diagnosed eight years prior to presentation and subsequent type two diabetes mellitus (T2DM), "
        "one prior episode of HTG-induced pancreatitis three years prior to presentation, and associated "
        "with an acute hepatitis, presented with a one-week history of polyuria, poor appetite, and vomiting."
    ),
    (
        "She was on metformin, glipizide, and dapagliflozin for T2DM and atorvastatin and gemfibrozil "
        "for HTG. She had been on dapagliflozin for six months at the time of presentation."
    ),
    (
        "Physical examination on presentation was significant for dry oral mucosa; significantly, her "
        "abdominal examination was benign with no tenderness, guarding, or rigidity. Pertinent laboratory "
        "findings on admission were: serum glucose 111 mg/dl, creatinine 0.4 mg/dL, triglycerides "
        "508 mg/dL, total cholesterol 122 mg/dL, and venous pH 7.27."
    ),
    (
        "The patient was diagnosed with stage III breast cancer. She started tamoxifen 20 mg daily. "
        "Her sister also had breast cancer. CT chest showed no metastasis. CT scan revealed a 2.8 cm "
        "right upper lobe nodule, highly suspicious for primary malignancy. Immediate PET-CT and "
        "biopsy strongly recommended."
    ),
]

data = spark.createDataFrame([[t] for t in sample_texts]).toDF("text")
data.show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                                text|
+----------------------------------------------------------------------------------------------------+
|Jennifer Smith is a 28-year-old female with a history of gestational diabetes mellitus diagnosed ...|
|She was on metformin, glipizide, and dapagliflozin for T2DM and atorvastatin and gemfibrozil for ...|
|Physical examination on presentation was significant for dry oral mucosa; significantly, her abdo...|
|The patient was diagnosed with stage III breast cancer. She started tamoxifen 20 mg daily. Her si...|
+----------------------------------------------------------------------------------------------------+



**Let's configure the generic model for all four task types and run the pipeline.**

In [16]:
zero_shot_clinical = (
    generic
    .setInputCols(["document"])
    .setOutputCol("extractions")
    .setEntityThreshold(0.4)
    # NER
    .setEntities([
        "problem",
        "treatment",
        "test",
        "body_part",
        "drug_dosage",
        "severity",
        "date",
    ])
    # Assertion via structures
    .setStructures([
        ("problem_info", [
            "text::str::medical condition, symptom, or diagnosis",
            "assertion::[present|absent|hypothetical|possible|conditional|associated_with_someone_else]",
        ]),
        ("treatment_info", [
            "text::str::drug, medication, procedure, or therapy",
            "assertion::[present|absent|hypothetical|possible|conditional|associated_with_someone_else]",
        ]),
    ])
    # Document classification
    .setClassifications([
        ("cancer_type",  ["lung", "breast", "colorectal", "other", "not_cancer"]),
        ("urgency",      ["routine", "urgent", "emergent"]),
    ])
    # Relation extraction
    .setRelations([
        "treatment_improves_problem",
        "treatment_causes_problem",
        "treatment_administered_for_problem",
        "test_reveals_problem",
        "test_conducted_for_problem",
    ])
)

clinical_pipe = nlp.Pipeline().setStages([document_assembler, zero_shot_clinical]).fit(data)
clinical_results = clinical_pipe.transform(data).cache()

**NER Chunks**

In [17]:
dfs_clinical = parse_multitask_results(clinical_results)

print("=" * 80)
print("NER CHUNKS")
print("=" * 80)
display(dfs_clinical["ner_chunks"])

NER CHUNKS


,idx,begin,end,chunk,sentence,ner_source,entity,confidence
0,0,97,113,eight years prior,0,extractions,date,0.49945667
1,0,202,212,HTG-induced,0,extractions,test,0.42177892
2,0,214,225,pancreatitis,0,extractions,problem,0.58654445
3,0,285,289,acute,0,extractions,severity,0.95193887
4,1,55,58,T2DM,0,extractions,test,0.8553094
5,1,64,75,atorvastatin,0,extractions,treatment,0.91268826
6,2,146,153,guarding,0,extractions,problem,0.9005587
7,2,232,240,111 mg/dl,0,extractions,severity,0.44999135
8,2,308,316,122 mg/dL,0,extractions,drug_dosage,0.48110217
9,2,323,331,venous pH,0,extractions,test,0.96852297


**Classifications**

In [18]:
print("=" * 80)
print("CLASSIFICATIONS")
print("=" * 80)
display(dfs_clinical["classifications"])

CLASSIFICATIONS


,idx,begin,end,result,sentence,category_type,task,confidence
0,0,0,376,other,0,classification,cancer_type,0.7181854
1,0,0,376,urgent,0,classification,urgency,0.5582844
2,1,0,174,colorectal,0,classification,cancer_type,0.3577044
3,1,0,174,routine,0,classification,urgency,0.78224075
4,2,0,337,not_cancer,0,classification,cancer_type,0.95115423
5,2,0,337,routine,0,classification,urgency,0.9832534
6,3,0,298,breast,0,classification,cancer_type,0.99992937
7,3,0,298,urgent,0,classification,urgency,0.76226383


**Relations**

In [19]:
print("=" * 80)
print("RELATIONS")
print("=" * 80)
display(dfs_clinical["relations"])

RELATIONS


,idx,begin,end,result,entity1,sentence,entity1_begin,entity2,chunk1_confidence,chunk1,chunk2,entity2_begin,entity1_end,category_type,entity2_end,chunk2_confidence
0,1,22,99,test_conducted_for_problem,head,0,22,tail,0.51473606,glipizide,HTG,97,30,relation,99,0.5739177
1,3,41,76,treatment_improves_problem,head,0,68,tail,0.8199601,tamoxifen,breast cancer,41,76,relation,53,0.67722845
2,3,41,76,treatment_causes_problem,head,0,68,tail,0.7451621,tamoxifen,breast cancer,41,76,relation,53,0.625967
3,3,41,76,treatment_administered_for_problem,head,0,68,tail,0.7857158,tamoxifen,breast cancer,41,76,relation,53,0.6635666


**Structured Extractions (Assertions)**

In [20]:
print("=" * 80)
print("STRUCTURED EXTRACTIONS (Assertions)")
print("=" * 80)
display(dfs_clinical["structures"])

STRUCTURED EXTRACTIONS (Assertions)


,idx,begin,end,result,text,confidence,start
0,0,0,300,problem_info,an acute hepatitis,0.545886,282
1,2,0,72,problem_info,dry oral mucosa,0.917725,57
2,3,0,155,problem_info,breast cancer. CT chest showed no metastasis,0.547246,111


## 3.2. Detailed Clinical NER with Entity Descriptions

In this section, we use the `LABEL::description` format to define a comprehensive set of clinical NER entities. Providing descriptions guides the model to differentiate between similar entity types more effectively. For example, `"drug::Drug or medication name like aspirin"` helps the model distinguish drug names from dosage values or routes of administration.

**Let's configure detailed NER entities with descriptions.**

In [21]:
zero_shot_ner = (
    generic
    .setInputCols(["document"])
    .setOutputCol("extractions")
    .setEntityThreshold(0.6)
    .setEntities([
        "date::Date of occurrence",
        "gender::Gender of patient like she or he",
        "patient_name::Patient Name like John Doe",
        "doctor_name::Doctor Name like Dr. Smith",
        "disorder::Disorder or medical condition",
        "symptom::Symptom or clinical finding",
        "treatment::Treatment or procedure",
        "test::Test or examination",
        "measurement::Measurement or lab value and results",
        "drug::Drug or medication name like aspirin",
        "dosage::Dosage of medication like 20 mg",
        "route::Route of medication like oral or intravenous",
        "frequency::Frequency of medication like daily or twice a day",
    ])
    .setClassifications([])
    .setRelations([])
    .setStructures([])
)

ner_pipe = nlp.Pipeline().setStages([document_assembler, zero_shot_ner]).fit(data)
ner_results = ner_pipe.transform(data).cache()

dfs_ner = parse_multitask_results(ner_results)
display(dfs_ner["ner_chunks"])

,idx,begin,end,chunk,sentence,ner_source,entity,confidence
0,0,0,13,Jennifer Smith,0,extractions,patient_name,0.9179745
1,0,20,30,28-year-old,0,extractions,date,0.7658821
2,0,32,37,female,0,extractions,gender,0.9999955
3,0,146,171,type two diabetes mellitus,0,extractions,disorder,0.6866514
4,0,339,346,polyuria,0,extractions,symptom,0.98058
5,1,11,19,metformin,0,extractions,treatment,0.83918035
6,1,81,91,gemfibrozil,0,extractions,drug,0.82100195
7,1,97,99,HTG,0,extractions,disorder,0.88231957
8,1,102,104,She,0,extractions,gender,0.9999529
9,2,57,71,dry oral mucosa,0,extractions,disorder,0.9764271
